# 7 — Combining the three regressions into one prediction

Notebook 6 stayed close to the Al-Aghbary et al. (2026) recipe: one quantile
regression forest, clustered into regional experts, with a spread of diagnostics
built on the forest's own quantiles. That is a faithful reproduction, and it is
also only one view of the problem.

This project actually has three independent predictors of geothermal heat flow:

* **QRF** — a quantile regression forest (19 observables). Gives five quantiles
  per cell and a full leaf distribution we can read entropy from.
* **GBM** — gradient-boosted quantile regression (14 observables) with a
  conformal calibration step. Gives a low/median/high triple.
* **SIM** — a similarity / analogue predictor (22 observables). Gives a median,
  a spread, and a per-cell histogram over binned heat-flow values.

They see different observables, make different modelling assumptions, and fail in
different places. That last point is the useful one. Where the three agree we can
be more confident than any single model would justify; where they disagree we are
looking straight at *structural* uncertainty — the part that comes from not
knowing which model form is right — and no single-model interval can express it.

This notebook does four things:

1. **Pools** the three sets of quantiles into one mixture distribution, rather
   than averaging their intervals. Averaging intervals throws away the shape;
   pooling keeps it.
2. **Weights** the methods by skill, using the corrected hold-out RMSE from
   notebook 4, so the weakest predictor cannot drag the median around.
3. **Decomposes the total variance into three parts** — aleatoric, within-method
   epistemic, and a new *between-method structural* term — so we can say *why* a
   cell is uncertain.
4. **Re-derives the confidence and explainability maps** on the pooled
   distribution, extending the paper's four explainability classes with a fifth
   that flags cells where the methods themselves disagree.

Everything feeds off notebook 6's artefacts and the corrected target grids from
5a/5b/5c. Shared paths, quantile levels, weighting mode and the PICP target all
live in `config.py`; nothing numerical is hard-coded here.


## Section 0 — Configuration and toggles

As in notebook 6, the first cell only sets switches and pulls shared constants
from `config.py`. The ensemble-specific knobs — the quantile grid, the weighting
mode, the methods to include and the PICP target — were added to `config.py` for
this notebook so notebooks 6 and 7 draw from the same source of truth.


In [ ]:
import sys, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)   # nanmean over empty slices etc.

from config import *          # paths, params, helpers (to_mW, ENSEMBLE_*, CLUSTER_DIR, ENSEMBLE_DIR)

# ── Toggles ──────────────────────────────────────────────────────────────────
RUN_PICP_CHECK = True         # validate/inflate the pooled 90% band on the reference set
SAVE_GRIDS     = True         # write the fused NetCDF grids to output/ensemble
MAKE_FIGS      = True         # render the summary figures

# ── Shared constants (all from config.py) ─────────────────────────────────────
METHODS      = ENSEMBLE_METHODS         # ["qrf", "gbm", "sim"]
QLEVELS      = ENSEMBLE_QUANTILES       # [0.05, 0.25, 0.50, 0.75, 0.95]
WEIGHT_MODE  = ENSEMBLE_WEIGHT_MODE     # "inv_rmse" | "equal"
PICP_TARGET  = ENSEMBLE_PICP_TARGET     # 0.90 nominal coverage of the 5-95 band
FIG_DIR      = fig_dir_ensemble
ENS_DIR      = ENSEMBLE_DIR

ENS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"methods       : {METHODS}")
print(f"quantiles     : {QLEVELS}")
print(f"weight mode   : {WEIGHT_MODE}")
print(f"PICP target   : {PICP_TARGET}")
print(f"ensemble dir  : {ENS_DIR}")
print(f"figure dir    : {FIG_DIR}")


## Section 1 — Load the hand-off from notebook 6 and the method skill scores

Notebook 6 wrote a single self-describing summary, `cluster_summary.json`, that
points at every artefact it produced: the chosen number of clusters, the
cross-validated diagnostic thresholds, the per-domain cluster labels and the
pickled per-cell diagnostics. We read that summary rather than re-deriving any of
it, so the two notebooks can never drift apart.

The method weights come from notebook 4's hold-out metrics
(`output/models/{method}_metrics.csv`). We use the **corrected** median RMSE for
each method -- the row after the spline bias correction -- because that is the
number that reflects how each predictor actually performs on unseen sites. Lower
RMSE earns a larger weight; the weights are normalised to sum to one.

Inverse-RMSE weighting is deliberately conservative. QRF and GBM share a lot of
information (both are tree ensembles on overlapping observables), so we are not
trying to be clever about redundancy here -- we simply down-weight the weaker
predictor. If a fully generalised-least-squares weighting is wanted later, the
correlation structure between the three residual series is the missing
ingredient; the hook is left in the markdown, not the code, so this notebook
stays reproducible.


In [ ]:
# --- 1a. notebook-6 hand-off -------------------------------------------------
summary = json.load(open(CLUSTER_DIR / "cluster_summary.json"))
NCLUSTERS_BEST = int(summary["NCLUSTERS_BEST"])
thresholds     = summary["cv_thresholds"]          # b_mean_cv, Rcv25, Rcv75, A50_cv, E50_cv, ...
obs_sel_nb6    = summary["obs_sel"]

print(f"notebook 6 used K={NCLUSTERS_BEST} clusters on {len(obs_sel_nb6)} features")
print("cv thresholds:")
for k, v in thresholds.items():
    print(f"    {k:14s} {v: .5g}")

# --- 1b. per-method skill -> weights -----------------------------------------
def corrected_rmse(method):
    """Median RMSE (mW/m2) of the spline-corrected model from notebook 4."""
    m = pd.read_csv(model_dir / f"{method}_metrics.csv")
    # prefer the row whose label contains 'corrected'; else fall back to the last row
    hit = m[m["label"].str.contains("corrected", case=False, na=False)]
    row = hit.iloc[0] if len(hit) else m.iloc[-1]
    return float(row["rmse_mW"])

rmse = {mth: corrected_rmse(mth) for mth in METHODS}

if WEIGHT_MODE == "inv_rmse":
    inv = {mth: 1.0 / rmse[mth] for mth in METHODS}
    total = sum(inv.values())
    weights = {mth: inv[mth] / total for mth in METHODS}
else:                                              # "equal"
    weights = {mth: 1.0 / len(METHODS) for mth in METHODS}

print("\nmethod   RMSE(mW)   weight")
for mth in METHODS:
    print(f"  {mth:4s}   {rmse[mth]:7.2f}    {weights[mth]:.3f}")
assert abs(sum(weights.values()) - 1.0) < 1e-9


## Section 2 — Load the three corrected target grids

Notebooks 5a, 5b and 5c wrote one NetCDF per method per region into
`output/targets/`, named `{region}_{METHOD}_v{MODEL_VERSION}.nc`. They all share
the same `(y, x)` grid, so we can stack them without regridding. We read the
**corrected** quantiles from each -- the spline-bias-corrected fields, which are
the ones the paper's diagnostics and our own comparison should use.

Each method contributes a different amount of shape information, and we keep all
of it:

* **QRF** gives the full five-point quantile function
  `qrf_q05/q25/q50/q75/q95_corr` plus a normalised leaf-entropy field.
* **GBM** gives a three-point function `gbm_q05/q50/q95_corr`. We interpolate the
  quartiles from those three points when we need a common five-point grid.
* **SIM** gives a median, a symmetric spread, and -- most usefully -- a per-cell
  histogram `sim_hist` over binned heat-flow values, which is a direct read on
  how multi-modal or spread-out the analogue set is.

Cells where any method is missing (open ocean, ice-shelf gaps) are carried as
NaN and excluded from every pooled statistic.


In [ ]:
import re

# ── Region prefixes ───────────────────────────────────────────────────────────
# Target files on disk are named like  Aq1_5_QRF_v0_4.nc / Aq15_GBM_v0_4.nc
# (Antarctica) and  Kq1_5_QRF_v0_4.nc / Kq15_GBM_v0_4.nc (Greenland). The prefix
# spelling varies by method, so we match a region by the leading letters only.
REGION_PREFIXES = {
    "ant": ("Aq", "ant"),   # Antarctica
    "grl": ("Kq", "grl"),   # Greenland
}

# match  <prefix>_?<METHOD>_?v<major>_<minor>.nc  with flexible separators.
_VER_RE = re.compile(
    r"^(?P<pre>.+?)_?(?P<method>QRF|GBM|SIM)_?v(?P<maj>\d+)_(?P<min>\d+)\.nc$",
    re.IGNORECASE,
)


def _latest_grid(prefixes, method):
    """Newest (path, (major, minor)) for one region+method, or (None, None)."""
    best_path, best_ver = None, None
    for p in sorted(targets_dir.glob("*.nc")):
        m = _VER_RE.match(p.name)
        if not m or m.group("method").upper() != method.upper():
            continue
        if not p.name.startswith(tuple(prefixes)):
            continue
        ver = (int(m.group("maj")), int(m.group("min")))
        if best_ver is None or ver > best_ver:
            best_path, best_ver = p, ver
    return best_path, best_ver


# ── Discover the latest version for every region + method ─────────────────────
RESOLVED = {}          # (region, method) -> path
_found_versions = set()
print(f"Scanning {targets_dir} for the latest target grids\n")
print(f"  {'region':6s} {'method':6s} {'version':8s} {'file'}")
print(f"  {'-'*6} {'-'*6} {'-'*8} {'-'*40}")
for region, prefixes in REGION_PREFIXES.items():
    for mth in METHODS:
        path, ver = _latest_grid(prefixes, mth)
        if path is None:
            print(f"  {region:6s} {mth.upper():6s} {'MISSING':8s} "
                  f"(no {prefixes[0]}..._{mth.upper()}_v*.nc found)")
            continue
        RESOLVED[(region, mth)] = path
        vtag = f"v{ver[0]}_{ver[1]}"
        _found_versions.add(vtag)
        print(f"  {region:6s} {mth.upper():6s} {vtag:8s} {path.name}")

# ── Pin the version used for the rest of the notebook ─────────────────────────
if _found_versions:
    RESOLVED_VERSION = sorted(_found_versions)[-1]           # newest tag seen
    print(f"\nUsing target-grid version: {RESOLVED_VERSION}")
    if len(_found_versions) > 1:
        print(f"  note: mixed versions on disk {sorted(_found_versions)} — "
              f"each method used its own newest file (see table above)")
else:
    RESOLVED_VERSION = None
    print("\nNo target grids found — run 5a/5b/5c to populate output/targets/.")


def load_method_grids(region_key):
    """Return {method: xr.Dataset} for one region, using the newest file discovered
    above for each method. Raises if any method is missing."""
    grids = {}
    for mth in METHODS:
        path = RESOLVED.get((region_key, mth))
        if path is None:
            raise FileNotFoundError(
                f"no {mth.upper()} target grid for region '{region_key}' in {targets_dir} "
                f"-- run notebook 5{'abc'[METHODS.index(mth)]} first")
        grids[mth] = xr.open_dataset(path)
    return grids


# ── Probe Antarctica so the notebook fails early and clearly if 5a/5b/5c not run
print()
try:
    _g = load_method_grids("ant")
    for mth, ds in _g.items():
        print(f"  {mth:4s}  {tuple(ds.dims.items())}  vars={list(ds.data_vars)[:6]}...")
        ds.close()
except FileNotFoundError as e:
    print("NOTE:", e)
    print("This notebook consumes the corrected target grids; run 5a/5b/5c to populate output/targets/.")


## Section 3 — Pooling the three predictions

The wrong way to combine three prediction intervals is to average their edges:
take the mean of the three 5th percentiles and the mean of the three 95th
percentiles. That collapses three differently-shaped distributions into one
symmetric box and quietly discards exactly the disagreement we care about.

The right way is to treat each method as giving a **distribution** for the cell,
and to build the mixture

\[
F_{\text{mix}}(h) \;=\; \sum_{m} w_m \, F_m(h),
\]

where \(F_m\) is method \(m\)'s cumulative distribution for the cell and \(w_m\)
is its weight from Section 1. The pooled quantiles are read straight off the
mixture CDF: the pooled median is the \(h\) at which \(F_{\text{mix}}(h)=0.5\),
the pooled 90% band runs from \(F_{\text{mix}}^{-1}(0.05)\) to
\(F_{\text{mix}}^{-1}(0.95)\), and so on.

Mixture pooling has two properties we want. First, the pooled band is *wider*
than any single method's band whenever the methods disagree about location -- it
inflates automatically in exactly the places where structural uncertainty is
largest. Second, it keeps skew and multi-modality: if two methods sit low and one
sits high, the pooled distribution is genuinely bimodal, and the median lands
between the modes rather than being smeared away.

**Implementation.** Each method already gives us its quantile function at a
handful of levels. We build a fine, shared heat-flow axis, turn each method's
quantiles into a piecewise-linear CDF on that axis (monotone by construction),
mix the CDFs with the method weights, and invert once to read off the pooled
levels in `QLEVELS`. This is done vectorised over all valid cells at once.


In [ ]:
# Common quantile representation for each method -----------------------------
# We standardise every method onto the 5 levels in QLEVELS. QRF supplies all 5
# directly; GBM supplies (0.05, 0.50, 0.95) and we linearly fill the quartiles;
# SIM supplies (median, std) which we turn into matching normal quantiles.
from scipy.stats import norm

def method_quantile_stack(ds, method):
    """Return array (n_levels, ny, nx) of this method's quantiles on QLEVELS."""
    if method == "qrf":
        names = {0.05: "qrf_q05_corr", 0.25: "qrf_q25_corr", 0.50: "qrf_q50_corr",
                 0.75: "qrf_q75_corr", 0.95: "qrf_q95_corr"}
        return np.stack([ds[names[q]].values for q in QLEVELS], axis=0)

    if method == "gbm":
        q05 = ds["gbm_q05_corr"].values
        q50 = ds["gbm_q50_corr"].values
        q95 = ds["gbm_q95_corr"].values
        # fill quartiles by linear interpolation in probability along each leg
        q25 = q05 + (q50 - q05) * ((0.25 - 0.05) / (0.50 - 0.05))
        q75 = q50 + (q95 - q50) * ((0.75 - 0.50) / (0.95 - 0.50))
        return np.stack([q05, q25, q50, q75, q95], axis=0)

    if method == "sim":
        med = ds["sim_q50_corr"].values
        std = ds["sim_std_corr"].values
        # symmetric quantiles from a normal with this median and spread; std==0
        # or NaN collapses to the median (a delta), which is the honest reading
        std_safe = np.where(np.isfinite(std) & (std > 0), std, 0.0)
        z = {q: float(norm.ppf(q)) for q in QLEVELS}
        return np.stack([med + z[q] * std_safe for q in QLEVELS], axis=0)

    raise ValueError(f"unknown method {method}")

print("quantile-stack helpers ready for:", METHODS)


In [ ]:
def pool_quantiles(method_stacks, weights, n_axis=400):
    """Quantile-mixture pooling.

    method_stacks : {method: array (n_levels, ny, nx)} of quantiles on QLEVELS
    weights       : {method: scalar}, summing to 1
    returns       : array (n_levels, ny, nx) of POOLED quantiles on QLEVELS

    Per cell we build a shared heat-flow axis, convert each method's quantile
    function to a CDF on that axis, mix, and invert. Vectorised over cells; the
    per-cell interpolation loop runs only over valid cells.
    """
    mths = list(method_stacks.keys())
    ny, nx = method_stacks[mths[0]].shape[1:]
    N = ny * nx
    ql = np.asarray(QLEVELS)

    flat = {m: method_stacks[m].reshape(len(QLEVELS), N) for m in mths}

    valid = np.ones(N, dtype=bool)
    for m in mths:
        valid &= np.all(np.isfinite(flat[m]), axis=0)

    pooled = np.full((len(QLEVELS), N), np.nan, dtype=np.float64)
    if not valid.any():
        return pooled.reshape(len(QLEVELS), ny, nx)

    vidx = np.where(valid)[0]
    lo = np.min(np.stack([flat[m][0,  vidx] for m in mths]), axis=0)
    hi = np.max(np.stack([flat[m][-1, vidx] for m in mths]), axis=0)
    span = np.where(hi > lo, hi - lo, 1e-9)
    lo = lo - 0.02 * span
    hi = hi + 0.02 * span

    t = np.linspace(0.0, 1.0, n_axis)
    axis = lo[None, :] + t[:, None] * (hi - lo)[None, :]     # (n_axis, Nvalid)

    mix_cdf = np.zeros((n_axis, vidx.size), dtype=np.float64)
    for m in mths:
        qm = flat[m][:, vidx]                                # (n_levels, Nvalid)
        for j in range(vidx.size):
            mix_cdf[:, j] += weights[m] * np.interp(
                axis[:, j], qm[:, j], ql, left=0.0, right=1.0)

    for li, q in enumerate(ql):
        for j in range(vidx.size):
            pooled[li, vidx[j]] = np.interp(q, mix_cdf[:, j], axis[:, j])

    return pooled.reshape(len(QLEVELS), ny, nx)

print("pool_quantiles ready")


## Section 4 — Where does the uncertainty come from?

Notebook 6 split the forest's variance into two parts, following the paper:
**aleatoric** (the spread inside the leaves, eq. 2) and **epistemic** (the spread
between trees, eq. 3), summing to the total (eq. 1). That is the right split for
a single model, but it has no way to express the uncertainty that comes from not
knowing *which model* to trust.

With three methods we can extend the law of total variance by one level. Treat
the choice of method as a random variable \(M\) with probabilities \(w_m\). Then
the variance of the pooled prediction \(H\) factors exactly as

\[
\underbrace{\operatorname{Var}(H)}_{\text{total}}
=\;
\underbrace{\mathbb{E}_M\!\big[\operatorname{Var}(H\mid M)\big]}_{\text{within-method}}
\;+\;
\underbrace{\operatorname{Var}_M\!\big(\mathbb{E}[H\mid M]\big)}_{\text{between-method (structural)}} .
\]

The first term is the weighted-average of each method's own variance. We can push
it one level further for the tree methods, because notebook 6 already gives us
their internal aleatoric/epistemic split, so the full three-way decomposition we
report is:

* **Aleatoric** \(V_a\) — irreducible scatter, the weighted mean of each method's
  within-model spread. This is noise in the heat-flow signal itself and does not
  shrink with more data or better models.
* **Epistemic (within-method)** \(V_e\) — the weighted mean of each method's
  parameter/sampling uncertainty (the between-tree term for QRF, the conformal
  spread for GBM, the analogue-set spread for SIM). This shrinks with more
  training data.
* **Structural (between-method)** \(V_s\) — the spread of the three method medians
  around the pooled median. This is the new term. It is large exactly where the
  methods tell different stories, and it is the honest signature of model-form
  uncertainty that no single model can see.

We estimate each method's own variance from its quantiles using the
semi-interquartile bandwidth from the paper (eq. 5), \(b=\tfrac12(Q_{75}-Q_{25})\),
which is a robust standard-deviation proxy; \(V \approx b^2\). For QRF, where the
notebook-6 aleatoric/epistemic split is available on the grid, we use it directly
so the reported \(V_a\) is not merely a quantile proxy.


In [ ]:
def semi_iqr_var(qstack):
    """Robust variance proxy from a quantile stack on QLEVELS.

    b = 0.5*(Q75 - Q25) is the semi-interquartile range (paper eq. 5); for a
    normal, b ~= 0.6745*sigma, so sigma ~= b/0.6745 and Var ~= (b/0.6745)**2.
    """
    i25, i75 = QLEVELS.index(0.25), QLEVELS.index(0.75)
    b = 0.5 * (qstack[i75] - qstack[i25])
    sigma = b / 0.674489750196
    return sigma ** 2

def decompose_variance(method_stacks, medians, pooled_median, weights,
                        qrf_aleatoric=None):
    """Three-term law-of-total-variance decomposition on the grid.

    method_stacks : {method: (n_levels, ny, nx)}   per-method quantiles
    medians       : {method: (ny, nx)}             per-method median (q50)
    pooled_median : (ny, nx)                        pooled q50 from Section 3
    weights       : {method: scalar}
    qrf_aleatoric : optional (ny, nx) aleatoric variance grid from notebook 6

    returns dict of grids: V_total, V_aleatoric, V_epistemic, V_structural
    """
    mths = list(method_stacks.keys())
    shape = pooled_median.shape

    # within-method total variance per method (semi-IQR proxy)
    v_within = {m: semi_iqr_var(method_stacks[m]) for m in mths}

    # weighted-average within-method variance = aleatoric + within-method epistemic
    V_within_mean = np.zeros(shape)
    for m in mths:
        V_within_mean = V_within_mean + weights[m] * v_within[m]

    # aleatoric: use notebook-6 QRF aleatoric where available, else a floor of
    # the smallest per-method within variance (the most confident method sets
    # the irreducible-noise scale)
    if qrf_aleatoric is not None:
        V_a = np.where(np.isfinite(qrf_aleatoric), qrf_aleatoric,
                       np.nanmin(np.stack([v_within[m] for m in mths]), axis=0))
    else:
        V_a = np.nanmin(np.stack([v_within[m] for m in mths]), axis=0)
    V_a = np.minimum(V_a, V_within_mean)          # aleatoric cannot exceed the mean within

    # within-method epistemic is the remainder of the within-method budget
    V_e = np.clip(V_within_mean - V_a, 0.0, None)

    # structural: weighted variance of the method medians about the pooled median
    V_s = np.zeros(shape)
    for m in mths:
        V_s = V_s + weights[m] * (medians[m] - pooled_median) ** 2

    V_total = V_a + V_e + V_s
    return {"V_total": V_total, "V_aleatoric": V_a,
            "V_epistemic": V_e, "V_structural": V_s}

print("variance decomposition ready")


## Section 5 — Robustness, confidence and explainability on the pooled distribution

Notebook 6 built three diagnostic maps from the single forest, following the
paper: a robustness score \(R=1-H_n\) from the normalised entropy (so \(R=1\) is
a sharp, confident prediction), a **confidence** classification from the band
width and robustness (eqs 6-10, including the over-/under-confident classes),
and an **explainability** classification that says *why* a cell is uncertain by
comparing its aleatoric and epistemic parts (eqs 11-14).

We rebuild all three on the pooled distribution and reuse notebook 6's
cross-validated thresholds so the two sets of maps are drawn on the same scale
and are directly comparable. Each method feeds the part of the signal it measures
best, and we add exactly one new category the single model cannot express.

**Robustness.** We combine three entropy-like signals into one normalised score:
the QRF leaf entropy (`shannon_H_q50_corr` from 5a), the SIM histogram entropy
(from `sim_hist` in 5c, a direct read on analogue spread), and a
**between-method agreement** term that is high when the three medians sit close
together. A cell is only robust when the models are individually sharp *and*
they agree.

**Confidence (eqs 6-10).** Same five paper classes and thresholds as notebook 6
— High / Moderate / Low, plus Over- and Under-confident — evaluated on the
pooled semi-IQR bandwidth and the fused robustness. Cells that match no rule stay
`Unclassified` (code 5) rather than being forced into a class.

**Explainability (eqs 11-14 + one new class).** The paper's four classes
cross-tabulate the aleatoric part \(V_a\) against the epistemic part \(V_e\)
around their cross-validated medians:

0. **Low uncertainty** — both small.
1. **Epistemic-dominated** — epistemic large, aleatoric small (a call for more
   data, not better physics).
2. **Aleatoric-dominated** — aleatoric large, epistemic small (irreducible
   scatter; more data will not help).
3. **High uncertainty** — both large.
4. **Structurally-uncertain** (new) — the between-method structural variance
   \(V_s\) dominates the total. This is the class the single forest cannot see:
   the methods disagree about *where* the answer sits, not merely how spread out
   it is. When \(V_s / V_{\text{total}} \ge\) `STRUCTURAL_SHARE_THRESHOLD` the
   cell takes this label regardless of the \(V_a\)/\(V_e\) quadrant, because in
   that regime the quadrant is not the real story.


In [ ]:
# category codes (mirror notebook 6, with a 5th explainability class) --------
CONF_HIGH, CONF_MOD, CONF_LOW, CONF_OVER, CONF_UNDER, CONF_NA = 0, 1, 2, 3, 4, 5
CONF_LABELS = ["High", "Moderate", "Low", "Overconfident", "Underconfident", "Unclassified"]
CONF_COLORS = ["#2ca25f", "#99d8c9", "#fc8d59", "#d7191c", "#ffd700", "#bdbdbd"]

EXPL_LOW, EXPL_EPIST, EXPL_ALEAT, EXPL_HIGH, EXPL_STRUCT = 0, 1, 2, 3, 4
EXPL_LABELS = ["Low uncertainty", "Epistemic-dominated", "Aleatoric-dominated",
               "High uncertainty", "Structurally-uncertain"]
EXPL_COLORS = ["#2166ac", "#abd9e9", "#fdae61", "#d73027", "#762a83"]

def hist_entropy_norm(hist):
    """Normalised Shannon entropy (0..1) of a per-cell histogram stack.

    hist : (ny, nx, n_bins) non-negative counts/masses -> (ny, nx) entropy.
    """
    p = hist.astype(np.float64)
    s = p.sum(axis=-1, keepdims=True)
    p = np.divide(p, s, out=np.zeros_like(p), where=s > 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        H = -np.nansum(np.where(p > 0, p * np.log(p), 0.0), axis=-1)
    Hmax = np.log(hist.shape[-1])
    return np.clip(H / Hmax, 0.0, 1.0)

def agreement_score(medians, pooled_median, band):
    """Between-method agreement in [0,1]: 1 when medians coincide, ->0 as their
    spread grows relative to the pooled band (scale-free)."""
    mths = list(medians.keys())
    spread = np.sqrt(np.mean(np.stack(
        [(medians[m] - pooled_median) ** 2 for m in mths]), axis=0))
    ref = np.nanmedian(band[np.isfinite(band)])
    scale = np.where(band > 0, band, ref)
    return np.clip(1.0 - spread / (scale + 1e-9), 0.0, 1.0)


In [ ]:
def fused_diagnostics(pooled, medians, var_parts, qrf_H, sim_H, thr):
    """Return {robustness, confidence, explainability, band, struct_share} grids.

    pooled    : (n_levels, ny, nx) pooled quantiles on QLEVELS
    medians   : {method: (ny,nx)} per-method medians
    var_parts : dict from decompose_variance (V_total/aleatoric/epistemic/structural)
    qrf_H     : (ny,nx) normalised QRF leaf entropy (0..1), or None
    sim_H     : (ny,nx) normalised SIM histogram entropy (0..1), or None
    thr       : notebook-6 cv thresholds dict
    """
    i25, i50, i75 = QLEVELS.index(0.25), QLEVELS.index(0.50), QLEVELS.index(0.75)
    band = 0.5 * (pooled[i75] - pooled[i25])              # pooled semi-IQR (eq. 5)
    pooled_med = pooled[i50]
    finite = np.isfinite(band)

    # --- fused robustness R = mean of (1-H_qrf), (1-H_sim), agreement --------
    parts = []
    if qrf_H is not None:
        parts.append(1.0 - np.clip(qrf_H, 0.0, 1.0))
    if sim_H is not None:
        parts.append(1.0 - np.clip(sim_H, 0.0, 1.0))
    parts.append(agreement_score(medians, pooled_med, band))
    R = np.nanmean(np.stack(parts), axis=0)

    # --- confidence (eqs 6-10), notebook-6 thresholds & rules ----------------
    conf = np.full(band.shape, CONF_NA, dtype=np.int8)
    b_mean = thr["b_mean_cv"]; R25 = thr["Rcv25"]; R75 = thr["Rcv75"]
    b_def  = thr.get("b_deficit_cv", b_mean); b_exc = thr.get("b_excess_cv", b_mean)
    conf[finite & (band <= b_mean) & (R >= R75)] = CONF_HIGH
    conf[finite & (band <= b_mean) & (R >= R25) & (R < R75)] = CONF_MOD
    conf[finite & (band <= b_mean) & (R <  R25)] = CONF_LOW
    conf[finite & (band <  b_def)  & (R <  R25)] = CONF_OVER
    conf[finite & (band >  b_exc)  & (R >= R75)] = CONF_UNDER

    # --- explainability (eqs 11-14) + structural override --------------------
    sa, se = var_parts["V_aleatoric"], var_parts["V_epistemic"]
    A50, E50 = thr["A50_cv"], thr["E50_cv"]
    expl = np.full(band.shape, -1, dtype=np.int8)
    expl[finite & (sa <= A50) & (se <= E50)] = EXPL_LOW
    expl[finite & (sa <= A50) & (se >  E50)] = EXPL_EPIST
    expl[finite & (sa >  A50) & (se <= E50)] = EXPL_ALEAT
    expl[finite & (sa >  A50) & (se >  E50)] = EXPL_HIGH

    struct_share = np.divide(var_parts["V_structural"], var_parts["V_total"],
                             out=np.full(band.shape, np.nan),
                             where=var_parts["V_total"] > 0)
    dominant = np.isfinite(struct_share) & (struct_share >= STRUCTURAL_SHARE_THRESHOLD)
    expl[dominant] = EXPL_STRUCT

    return {"robustness": R, "confidence": conf, "explainability": expl,
            "band": band, "struct_share": struct_share}

print("fused diagnostics ready")


## Section 6 — Building the fused product

This is the driver. For each region it:

1. loads the three corrected target grids (Section 2);
2. turns each method into a five-point quantile stack on the common levels
   (Section 3);
3. pools them into one mixture distribution and reads the pooled quantiles;
4. decomposes the pooled variance into aleatoric / epistemic / structural parts
   (Section 4), passing notebook 6's QRF aleatoric grid through where it exists;
5. reads the QRF leaf entropy and the SIM histogram entropy and computes the
   fused robustness, confidence and explainability maps (Section 5);
6. assembles everything into one `xarray.Dataset` on the shared `(y, x)` grid.

The per-cell CDF inversion in the pooling step is the only heavy loop; it runs
over valid cells only and is fine at 25 km. Nothing here is hard-coded — the
methods, quantiles, weighting and thresholds all come from `config.py` and
notebook 6.


In [ ]:
def build_region(region_key):
    """Return an xarray.Dataset of the fused product for one region."""
    grids = load_method_grids(region_key)                   # {method: ds}
    ref   = grids[METHODS[0]]
    coords = {"y": ref["y"].values, "x": ref["x"].values}

    # 1. per-method quantile stacks on QLEVELS
    stacks  = {m: method_quantile_stack(grids[m], m) for m in METHODS}
    medians = {m: stacks[m][QLEVELS.index(0.50)] for m in METHODS}

    # 2. quantile-mixture pooling
    pooled = pool_quantiles(stacks, weights)                # (n_levels, ny, nx)

    # 3. variance decomposition (QRF aleatoric from notebook 6 if present on grid)
    qrf_ale = None
    for cand in ("qrf_var_aleatoric_corr", "variance_aleatoric", "qrf_aleatoric"):
        if cand in grids["qrf"]:
            qrf_ale = grids["qrf"][cand].values
            break
    var_parts = decompose_variance(stacks, medians, pooled[QLEVELS.index(0.50)],
                                   weights, qrf_aleatoric=qrf_ale)

    # 4. entropy inputs for robustness
    qrf_H = grids["qrf"]["shannon_H_q50_corr"].values if "shannon_H_q50_corr" in grids["qrf"] else None
    sim_H = hist_entropy_norm(grids["sim"]["sim_hist"].values) if "sim_hist" in grids["sim"] else None

    # 5. fused diagnostics
    diag = fused_diagnostics(pooled, medians, var_parts, qrf_H, sim_H, thresholds)

    # 6. assemble dataset
    def da(arr):
        return xr.DataArray(np.asarray(arr, dtype=np.float32), dims=["y", "x"], coords=coords)

    dv = {}
    qn = {0.05: "ens_q05", 0.25: "ens_q25", 0.50: "ens_q50", 0.75: "ens_q75", 0.95: "ens_q95"}
    for li, q in enumerate(QLEVELS):
        dv[qn[q]] = da(pooled[li])
    dv["ens_band"]          = da(diag["band"])
    dv["ens_var_total"]     = da(var_parts["V_total"])
    dv["ens_var_aleatoric"] = da(var_parts["V_aleatoric"])
    dv["ens_var_epistemic"] = da(var_parts["V_epistemic"])
    dv["ens_var_structural"]= da(var_parts["V_structural"])
    dv["ens_struct_share"]  = da(diag["struct_share"])
    dv["ens_robustness"]    = da(diag["robustness"])
    dv["ens_confidence"]    = da(diag["confidence"].astype(np.float32))
    dv["ens_explainability"]= da(diag["explainability"].astype(np.float32))
    for m in METHODS:
        dv[f"{m}_q50_in"] = da(medians[m])                  # method medians for reference

    ds = xr.Dataset(dv, coords=coords)
    ds.attrs.update(
        methods=",".join(METHODS),
        weights=",".join(f"{m}:{weights[m]:.4f}" for m in METHODS),
        weight_mode=WEIGHT_MODE,
        quantiles=",".join(str(q) for q in QLEVELS),
        model_version=str(MODEL_VERSION),
        note="7_ENSEMBLE quantile-mixture pooling of qrf/gbm/sim; heat flow in W/m2",
    )
    for m in METHODS:
        grids[m].close()
    return ds

products = {}
for key in REGION_PREFIXES:
    try:
        products[key] = build_region(key)
        d = products[key]
        n_valid = int(np.isfinite(d["ens_q50"].values).sum())
        print(f"{key}: fused dataset {dict(d.dims)}  valid cells={n_valid}")
    except FileNotFoundError as e:
        print(f"{key}: SKIPPED -- {e}")


## Section 7 — Does the pooled band cover the truth as often as it claims?

A 90% interval should contain the true value about 90% of the time. The
prediction-interval coverage probability (PICP) measures this directly: sample
the pooled interval at the sites where we actually have a measured heat-flow
value, and count how often the measurement falls inside the pooled 5-95 band.

Two things can go wrong. If PICP is well below the nominal `ENSEMBLE_PICP_TARGET`
the band is **overconfident** and we widen it; if it is well above, the band is
conservative, which we leave alone because a slightly wide band is the safe error
to make for a hazard-style product.

We calibrate with a single multiplicative factor \(\lambda \ge 1\) applied
symmetrically about the pooled median,
\(q'_{05}=q_{50}-\lambda(q_{50}-q_{05})\) and
\(q'_{95}=q_{50}+\lambda(q_{95}-q_{50})\), choosing the smallest \(\lambda\) that
brings PICP up to the target. This keeps the median untouched and only inflates
the tails, which is exactly where mixture pooling can still be too tight if the
three methods happen to agree by luck rather than by signal.

The reference measurements live at scattered points, so we read the pooled
quantiles at the nearest grid cell to each site. If the reference file or a
region's grid is not available in this checkout, the cell reports the per-method
PICP already recorded in `output/models/*_metrics.csv` and skips inflation, so
the notebook still runs end to end.


In [ ]:
def sample_grid_at_points(ds, var, xs, ys):
    """Nearest-cell lookup of `var` at scattered (xs, ys) on ds's (y,x) grid."""
    da = ds[var]
    return da.sel(x=xr.DataArray(xs, dims="pt"),
                  y=xr.DataArray(ys, dims="pt"), method="nearest").values

def picp_from_metrics():
    """Fallback: report the per-method PICP already computed in notebook 4."""
    rows = []
    for m in METHODS:
        df = pd.read_csv(model_dir / f"{m}_metrics.csv")
        hit = df[df["label"].str.contains("corrected", case=False, na=False)]
        row = hit.iloc[0] if len(hit) else df.iloc[-1]
        rows.append((m, float(row.get("picp", np.nan))))
    return rows

lam = {key: 1.0 for key in products}          # inflation factor per region
if RUN_PICP_CHECK:
    ref_ok = parquet_ref.exists()
    if not ref_ok:
        print("reference observations not found; reporting per-method PICP from notebook 4:")
        for m, p in picp_from_metrics():
            print(f"    {m:4s}  PICP={p:.3f}")
        print(f"pooled-band inflation skipped (target {PICP_TARGET}). Re-run with data present.")
    else:
        ref = pd.read_parquet(parquet_ref)
        # column guesses for coords + target; adjust here if your schema differs
        xcol = next((c for c in ("x","X","easting","POINT_X") if c in ref.columns), None)
        ycol = next((c for c in ("y","Y","northing","POINT_Y") if c in ref.columns), None)
        hcol = next((c for c in ("hf","HF","heat_flow","q","GHF","hf_Wm2") if c in ref.columns), None)
        for key, ds in products.items():
            if not (xcol and ycol and hcol):
                print(f"{key}: reference schema not recognised; skipping PICP.")
                continue
            xs = ref[xcol].to_numpy(); ys = ref[ycol].to_numpy()
            truth = ref[hcol].to_numpy()
            lo = sample_grid_at_points(ds, "ens_q05", xs, ys)
            hi = sample_grid_at_points(ds, "ens_q95", xs, ys)
            med = sample_grid_at_points(ds, "ens_q50", xs, ys)
            ok = np.isfinite(lo) & np.isfinite(hi) & np.isfinite(truth)
            if ok.sum() < 20:
                print(f"{key}: too few in-domain reference points ({ok.sum()}); skipping.")
                continue
            inside = (truth[ok] >= lo[ok]) & (truth[ok] <= hi[ok])
            picp0 = inside.mean()
            # smallest lambda>=1 reaching the target (grid search)
            L = 1.0
            if picp0 < PICP_TARGET:
                for cand in np.linspace(1.0, 3.0, 41):
                    lo_c = med[ok] - cand * (med[ok] - lo[ok])
                    hi_c = med[ok] + cand * (hi[ok] - med[ok])
                    if ((truth[ok] >= lo_c) & (truth[ok] <= hi_c)).mean() >= PICP_TARGET:
                        L = float(cand); break
                else:
                    L = 3.0
            lam[key] = L
            picp_final = (( truth[ok] >= med[ok]-L*(med[ok]-lo[ok])) &
                          ( truth[ok] <= med[ok]+L*(hi[ok]-med[ok])) ).mean()
            print(f"{key}: n={ok.sum():4d}  PICP={picp0:.3f} -> lambda={L:.2f} -> PICP={picp_final:.3f} "
                  f"(target {PICP_TARGET})")


In [ ]:
# apply the inflation factor to the stored band and 5/95 quantiles ----------
for key, ds in products.items():
    L = lam[key]
    if L <= 1.0:
        continue
    med = ds["ens_q50"]
    ds["ens_q05"] = med - L * (med - ds["ens_q05"])
    ds["ens_q95"] = med + L * (ds["ens_q95"] - med)
    ds["ens_band"] = ds["ens_band"] * L
    ds.attrs["picp_inflation"] = f"{key}:{L:.3f}"
    print(f"{key}: applied lambda={L:.2f} to pooled band and tails")
if all(v <= 1.0 for v in lam.values()):
    print("no inflation needed (or PICP check skipped).")


## Section 8 — Maps of the fused product

Four figures per region, drawn with the shared colour maps from `config.py` so
they sit alongside the notebook-6 panels without a restyle:

* **Fused prediction and band** — the pooled median heat flow and the pooled
  90% band width, both in mW m⁻².
* **Variance budget** — aleatoric, within-method epistemic and between-method
  structural variance side by side, on a shared scale, so the reader can see at a
  glance *which* kind of uncertainty dominates where.
* **Structural share** — the single map that is new to this notebook: the
  fraction of the total variance that comes from the methods disagreeing. Bright
  cells are where the ensemble is telling you something a single model cannot.
* **Confidence and explainability** — the categorical maps, with the extra
  structurally-uncertain class shown in purple.

Heat-flow fields are converted to mW m⁻² with `to_mW`; variance is in
(W m⁻²)² and converted to (mW m⁻²)² by `to_mW` applied twice, matching notebook 6.


In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap

def _extent(ds):
    x = ds["x"].values; y = ds["y"].values
    return [x.min(), x.max(), y.min(), y.max()]

def _show(ax, ds, arr, cmap, title, vmin=None, vmax=None):
    im = ax.imshow(np.asarray(arr), origin="upper", extent=_extent(ds),
                   cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    return im

def figure_prediction(key, ds):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
    im0 = _show(axes[0], ds, to_mW(ds["ens_q50"].values), hf_cmap,
                "(a) Fused median heat flow")
    fig.colorbar(im0, ax=axes[0], shrink=0.8, label="mW m$^{-2}$")
    im1 = _show(axes[1], ds, to_mW(ds["ens_band"].values), unc_cmap,
                "(b) Pooled 90% band (semi-IQR)")
    fig.colorbar(im1, ax=axes[1], shrink=0.8, label="mW m$^{-2}$")
    fig.suptitle(f"{key.upper()} — fused prediction (quantile-mixture, 25 km)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{key}_ens_prediction{FIG_EXT}", dpi=FIG_DPI, bbox_inches="tight")
    return fig

def figure_variance(key, ds):
    parts = [("ens_var_aleatoric", "(a) Aleatoric"),
             ("ens_var_epistemic", "(b) Epistemic (within-method)"),
             ("ens_var_structural","(c) Structural (between-method)")]
    vmax = np.nanpercentile(to_mW(to_mW(ds["ens_var_total"].values)), 98)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (var, ttl) in zip(axes, parts):
        im = _show(ax, ds, to_mW(to_mW(ds[var].values)), unc_cmap, ttl, vmin=0, vmax=vmax)
    fig.colorbar(im, ax=axes, shrink=0.7, label="variance (mW m$^{-2}$)$^2$")
    fig.suptitle(f"{key.upper()} — variance budget (law of total variance)")
    fig.savefig(FIG_DIR / f"{key}_ens_variance_budget{FIG_EXT}", dpi=FIG_DPI, bbox_inches="tight")
    return fig

def figure_structural(key, ds):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = _show(ax, ds, ds["ens_struct_share"].values, "cmc.batlow",
               "Structural share of total variance", vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, shrink=0.8, label="$V_s / V_{total}$")
    fig.savefig(FIG_DIR / f"{key}_ens_structural_share{FIG_EXT}", dpi=FIG_DPI, bbox_inches="tight")
    return fig

def figure_categories(key, ds):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
    for ax, var, labels, colors, ttl in [
        (axes[0], "ens_confidence",     CONF_LABELS, CONF_COLORS, "(a) Confidence"),
        (axes[1], "ens_explainability", EXPL_LABELS, EXPL_COLORS, "(b) Explainability")]:
        cmap = ListedColormap(colors)
        norm = BoundaryNorm(np.arange(-0.5, len(labels) + 0.5), cmap.N)
        arr = ds[var].values.copy()
        arr[arr < 0] = np.nan                       # unset -> transparent
        im = ax.imshow(arr, origin="upper", extent=_extent(ds), cmap=cmap,
                       norm=norm, aspect="equal", interpolation="nearest")
        ax.set_title(ttl, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
        cb = fig.colorbar(im, ax=ax, shrink=0.8, ticks=range(len(labels)))
        cb.ax.set_yticklabels(labels, fontsize=8)
    fig.suptitle(f"{key.upper()} — confidence and explainability (fused)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{key}_ens_confidence_explain{FIG_EXT}", dpi=FIG_DPI, bbox_inches="tight")
    return fig

if MAKE_FIGS:
    for key, ds in products.items():
        figure_prediction(key, ds)
        figure_variance(key, ds)
        figure_structural(key, ds)
        figure_categories(key, ds)
        plt.show()
    print(f"figures written to {FIG_DIR}")
else:
    print("figures skipped (MAKE_FIGS=False)")


## Section 9 — Write the grids and summarise

In [ ]:
if SAVE_GRIDS:
    manifest = {"model_version": str(MODEL_VERSION), "methods": METHODS,
                "weight_mode": WEIGHT_MODE, "weights": weights,
                "quantiles": QLEVELS, "picp_target": PICP_TARGET,
                "picp_inflation": {k: float(v) for k, v in lam.items()},
                "structural_share_threshold": STRUCTURAL_SHARE_THRESHOLD,
                "regions": {}}
    enc = {v: {"zlib": True, "complevel": netcdf_compression_level}
           for k, ds in products.items() for v in ds.data_vars}
    for key, ds in products.items():
        out = ENS_DIR / f"{key}_ensemble_v{MODEL_VERSION}.nc"
        ds.to_netcdf(out, encoding={v: enc[v] for v in ds.data_vars})
        manifest["regions"][key] = {
            "grid": str(out),
            "mean_ghf_mW":   round(float(to_mW(np.nanmean(ds["ens_q50"].values))), 2),
            "mean_band_mW":  round(float(to_mW(np.nanmean(ds["ens_band"].values))), 2),
            "mean_struct_share": round(float(np.nanmean(ds["ens_struct_share"].values)), 3),
        }
        print(f"{key}: wrote {out}")
    json.dump(manifest, open(ENS_DIR / "ensemble_summary.json", "w"), indent=2)
    print(f"wrote {ENS_DIR/'ensemble_summary.json'}")
else:
    print("grid saving skipped (SAVE_GRIDS=False)")


## Section 10 — Reading the maps as geology and glaciology

The point of the three benefits behind this project — better data, new methods,
and a qualitative read — is that the third one is where the first two earn their
keep. The fused maps are only useful if a geologist and a glaciologist can look
at them and recognise the ground. A few things to look for.

**East Antarctic craton.** Over the old cratonic interior — Dronning Maud Land
through to the Gamburtsev Subglacial Mountains and the Aurora and Wilkes basins —
the pooled median should sit low, in the low-to-mid 40s mW m⁻², and the pooled
band should be narrow. This is the easy case: thick, cold, ancient lithosphere,
and all three methods have plenty of cratonic analogues to lean on, so the
structural share stays low. Where it does *not* stay low, the usual culprit is a
concealed basin or a rift arm that one method's observables pick up and another's
do not.

**West Antarctic Rift System and the Antarctic Peninsula.** This is where the
ensemble matters most. Thinned lithosphere, young extension, and in places
suspected magmatic activity (Marie Byrd Land, the Thwaites–Pine Island sector)
push the median up and the band wide. Expect the structural class to light up
here: the tree methods lean on the potential-field and seismic observables while
the analogue method leans on its nearest matches, and in a tectonically young,
poorly-sampled region they genuinely disagree. That disagreement is not a defect
of the maps — it is the honest statement that the data cannot yet decide, and it
is exactly the signal an ice-sheet modeller needs before trusting a single
basal-heat number under a fast-changing glacier.

**Subglacial basins and the ice interface.** High heat flow under thick, wet-based
ice is the combination that matters for basal melt and ice dynamics. The places
where the fused median is high *and* the confidence map reads High are the ones
worth carrying into an ice-sheet model as a firm boundary condition. The places
where the median is high but the explainability map reads Structurally-uncertain
are the ones to carry as a *range*, not a number, and to flag for targeted
survey. The band width, in mW m⁻², is the quantity to hand the modeller.

**Greenland.** The Iceland hotspot track beneath central-east Greenland is the
one feature where a physically-motivated high should appear, and it is a good
sanity check: if the median does not lift along the reconstructed plume path, the
observables driving the models are not carrying that signal and the maps should
be read cautiously there. Around the margins, where borehole and marine
heat-flow control is better, expect narrow bands and High confidence.

**How to use the uncertainty, in one line.** Trust the median where confidence is
High and the structural share is low; treat the median as the centre of a genuine
range where the structural share is high; and read the explainability class to
know whether the fix is more data (epistemic), better physics (structural), or
neither because the scatter is real (aleatoric).


## Section 11 — What this notebook produced

For each region, `output/ensemble/{region}_ensemble_v{MODEL_VERSION}.nc` now holds:

* `ens_q05 … ens_q95` — the pooled quantiles (the fused best prediction is
  `ens_q50`; the robust range is `ens_q05`–`ens_q95`);
* `ens_band` — the pooled semi-interquartile bandwidth (mW-scale via `to_mW`);
* `ens_var_total` and its three parts `ens_var_aleatoric`,
  `ens_var_epistemic`, `ens_var_structural`, plus `ens_struct_share`;
* `ens_robustness`, `ens_confidence`, `ens_explainability` — the fused
  diagnostic maps, on the same scale as notebook 6;
* `{qrf,gbm,sim}_q50_in` — the three input medians, kept for traceability.

`output/ensemble/ensemble_summary.json` records the weights, the PICP inflation
factors and the per-region means, so a downstream notebook (or the paper's figure
scripts) can pick up the product without re-reading the grids.

Relative to notebook 6, the ensemble adds three things the single forest could
not give: a prediction that reflects all three methods weighted by skill, a band
that widens honestly where the methods disagree, and a structural-uncertainty
map that separates *not enough data* from *not sure which model is right*. That
last distinction is the scientific contribution of benefit B, and it is what
makes the maps safe to hand to an ice-sheet modeller.
